# DP2 — Extract Consolidated Visit Table for DDF from Butler\n
\n
**Author:** dagoret  \n
**Creation Date:** 2026-07-24  \n
**Last Date:** 2026-07-24  \n
**Version:** v1.0\n
\n
## Purpose\n
\n
Extract the **consolidated visit table** from Butler for DP2 (Data Preview 2) with all visit details\n
required for supernova simulation with `skysurvey` package. This includes:\n
- Visit identification (visitId, obs_id, day_obs, seq_num)\n
- Pointing information (ra, dec, skyRotation, azimuth, altitude, zenithDistance)\n
- Timing information (expMidptMJD, obsStartMJD, exposure time)\n
- Observing conditions (seeing, sky background, airmass)\n
- Photometric information (fiveSigmaDepth / limiting magnitude, band)\n
- Target/science program (to filter for DDF only)\n
\n
The final table is filtered to include **only Deep Drilling Field (DDF) visits** and saved\n
in a format compatible with `skysurvey.LSST` for SN Ia population simulation.\n
\n
## DDF Fields in DP2\n
\n
The DP2 Deep Drilling Fields include:\n
- COSMOS: (RA=150.119, Dec=+2.206)\n
- ECDFS: (RA=53.125, Dec=-28.100)\n
- ELAIS-S1: (RA=9.450, Dec=-44.000)\n
- XMM-LSS: (RA=35.708, Dec=-4.750)\n
- EDFS-a: (RA=58.900, Dec=-49.315)\n
- EDFS-b: (RA=63.600, Dec=-47.600)\n
- EDFS: (RA=61.240, Dec=-48.423)\n
- M49: (RA=187.400, Dec=+8.000)\n
\n
These are identified via the `science_program` or `target` columns containing 'DDF'\n
or specific field names (COSMOS, ELAIS-S1, XMM-LSS, ECDFS, EDFS, M49).

---\n
## 0. Imports

In [ ]:
# Standard library\n
import os\n
import sys\n
import warnings\n
import traceback\n
import gc\n
from pathlib import Path\n
\n
# Scientific stack\n
import numpy as np\n
import pandas as pd\n
import matplotlib\n
import matplotlib as mpl\n
import matplotlib.pyplot as plt\n
\n
# Astropy\n
from astropy.coordinates import SkyCoord\n
from astropy.table import Table\n
import astropy.units as u\n
from astropy.time import Time\n
from astropy.io import ascii as astropy_ascii\n
\n
# LSST stack\n
import lsst\n
from lsst.daf.butler import Butler\n
from lsst.geom import SpherePoint, degrees\n
import lsst.geom as geom\n
\n
# Version info\n
print(f\
)\n
print(f\
)\n
print(f\
)\n
print(f\
)\n
print(f\
)\n
print(\
)

In [ ]:
# Configure matplotlib for better readability\n
mpl.rcParams.update(\n
    {\n
        'figure.figsize': (10, 6),\n
        'font.size': 14,\n
        'axes.titlesize': 18,\n
        'axes.labelsize': 16,\n
        'xtick.labelsize': 14,\n
        'ytick.labelsize': 14,\n
        'legend.fontsize': 14,\n
        'legend.title_fontsize': 15,\n
        'figure.titlesize': 20,\n
    }\n
)\n
print(\
)

In [ ]:
# Suppress warnings for cleaner output\n
warnings.filterwarnings('ignore', category=FutureWarning)\n
warnings.filterwarnings('ignore', category=UserWarning)\n
warnings.filterwarnings('ignore', category=DeprecationWarning)

---\n
## 1. Configuration

In [ ]:
# Notebook tag and output directories\n
NB_TAG = \
\n
DIR_DATA = f\
\n
DIR_FIGS = f\
\n
os.makedirs(DIR_DATA, exist_ok=True)\n
os.makedirs(DIR_FIGS, exist_ok=True)\n
print(f\
)\n
print(f\
)\n
\n
# Output filenames\n
OUTPUT_VISITS_ECSV = os.path.join(DIR_DATA, \
)\n
OUTPUT_VISITS_PARQUET = os.path.join(DIR_DATA, \
)\n
print(f\
)\n
print(f\
)

In [ ]:
# Butler configuration for DP2\n
REPO = \
\n
\n
# Collections to query - DP2 processed collections\n
COLLECTIONS = [\n
    \
,\n
    \
,\n
    \
,\n
    \
,\n
    \
,\n

In [ ]:
# DDF field coordinates for reference\n
DDF_COORDS = {\n
    \
: (150.119, +2.206),\n
    \
: (53.125, -28.100),\n
    \
: (9.450, -44.000),\n
    \
: (35.708, -4.750),\n
    \
: (58.900, -49.315),\n
    \
: (63.600, -47.600),\n
    \
: (61.240, -48.423),\n
    \
: (187.400, +8.000),\n
}\n
\n
print(\
)\n
for field, (ra, dec) in DDF_COORDS.items():\n
    print(f\
)

---\n
## 2. Butler Connection

In [ ]:
# Connect to Butler\n
try:\n
    butler = Butler(REPO, collections=COLLECTIONS)\n
    registry = butler.registry\n
    print(\
)\n
except Exception as e:\n
    print(f\
)\n
    traceback.print_exc()\n
    sys.exit(1)

In [ ]:
# Get SkyMap\n
try:\n
    skymap = butler.get(\
, skymap=SKYMAP_NAME, collections=COLLECTIONS)\n
    print(f\
)\n
except Exception as e:\n
    print(f\
)\n
    try:\n
        skymap = butler.get(\
, skymap=SKYMAP_NAME)\n
        print(f\
)\n
    except Exception as e2:\n
        print(f\
)

---\n
## 3. Discover Dataset Types

In [ ]:
# Query all dataset types\n
all_datasets = list(registry.queryDatasetTypes())\n
print(f\
)\n
\n
# Filter for visit/exposure related datasets\n

In [ ]:
# Look for consolidated visit tables\n
consolidated_tables = [dt.name for dt in all_datasets \\\n
                       if any(kw in dt.name.lower() \\\n

---\n
## 4. Extract Visits from Exposure Records

In [ ]:
# Define WHERE clause for filtering\n
WHERE_CLAUSE_INSTRUMENT = f\
\n
WHERE_CLAUSE_DATE = f\
\n
print(f\
)

In [ ]:
def extract_visits_from_exposure(registry, where_clause, limit=None):\n
    \
\
    Extract visit information from exposure dimension records in Butler registry.\n
    \n
    This queries the 'exposure' dimension which contains detailed information about\n
    each exposure including pointing, timing, observation metadata, and exposure parameters.\n
    \n
    Parameters\n
    ----------\n
    registry : Butler registry\n
        The Butler registry to query\n
    where_clause : str\n
        SQL-like WHERE clause for filtering exposures\n
    limit : int, optional\n
        Maximum number of records to return (None for all)\n
    \n
    Returns\n
    -------\n
    df_visits : pandas.DataFrame\n
        DataFrame containing visit information with columns:\n
        - id: exposure ID (visit ID)\n
        - obs_id: observation ID\n
        - day_obs: day of observation (YYYYMMDD format)\n
        - seq_num: sequence number\n
        - ra: right ascension (degrees)\n
        - dec: declination (degrees)\n
        - sky_angle: sky rotation angle (degrees)\n
        - azimuth: azimuth (degrees)\n
        - zenith_angle: zenith angle (degrees)\n
        - airmass: calculated airmass\n
        - filter: physical filter name\n
        - band: single-letter band identifier (u, g, r, i, z, y)\n
        - observation_type: type of observation (science, calibration, etc.)\n
        - science_program: science program identifier\n
        - target_name: target field name\n
        - exposure_time: exposure duration (seconds)\n
        - mjd: midpoint Modified Julian Date\n
    \
\
    \n

In [ ]:
# Extract all science visits from Butler\n
print(\
)\n
df_visits = extract_visits_from_exposure(registry, WHERE_CLAUSE_DATE)\n
print(f\
)\n
print(f\

In [ ]:
# Show first few rows\n
print(\
)\n
display(df_visits.head(3))

---\n
## 5. Extract Image Quality Metrics

In [ ]:
# Check for cp_pipe_cpVisitTable which contains consolidated visit metrics\n
CP_VISIT_TABLE_TYPE = 'cp_pipe_cpVisitTable'\n
\n

In [ ]:
def extract_metrics_from_visit_table(butler, dataset_type_name, where_clause, limit=1000):\n
    \
\
    Extract metrics from consolidated visit table (cp_pipe_cpVisitTable).\n
    \n
    The cp_pipe_cpVisitTable contains per-visit consolidated metrics including:\n
    - mean_seeing: mean seeing FWHM (arcseconds)\n
    - mean_skyBg: mean sky background (mag/arcsec^2)\n
    - mean_maglim: mean 5-sigma limiting magnitude\n
    - fiveSigmaDepth: limiting depth\n
    \n
    Parameters\n
    ----------\n
    butler : Butler\n
    dataset_type_name : str\n
    where_clause : str\n
    limit : int\n
    \n
    Returns\n
    -------\n
    metrics_dict : dict\n
        Dictionary mapping visit IDs to their metrics\n
    \
\
    metrics_dict = {}\n
    \n
    try:\n
        # Query datasets\n
        dataset_refs = list(registry.queryDatasets(\n
            datasetType=dataset_type_name,\n
            where=where_clause,\n
            limit=limit\n
        ))\n
        \n
        print(f\
)\n
        \n
        if not dataset_refs:\n
            return metrics_dict\n
        \n
        # Fetch data\n
        try:\n

In [ ]:
# Extract metrics from cp_pipe_cpVisitTable\n
print(\
)\n
metrics_dict = extract_metrics_from_visit_table(butler, CP_VISIT_TABLE_TYPE, WHERE_CLAUSE_DATE, limit=5000)\n
print(f\
)\n
\n
if metrics_dict:\n
    print(\
)\n

---\n
## 6. Filter for DDF Visits Only

In [ ]:
# Define DDF identification patterns\n
DDF_PATTERNS = [\n
    'DDF',\n
    'COSMOS',\n
    'ELAIS',\n
    'XMM',\n
    'ECDFS',\n
    'EDFS',\n
    'M49',\n

In [ ]:
# Show DDF statistics\n
print(\
)\n

In [ ]:
# Merge metrics into DDF visits\n
if metrics_dict:\n
    # Create DataFrame from metrics\n

---\n
## 7. Calculate Missing Metrics

In [ ]:
# Ensure airmass is calculated\n

In [ ]:
# Standardize column names for skysurvey compatibility\n
column_mapping = {\n
    'expMidptMJD': 'mjd',\n
    'tracking_ra': 'ra',\n
    'tracking_dec': 'dec',\n
}\n
\n
for new_name, old_name in column_mapping.items():\n
    if old_name in df_ddf.columns and new_name not in df_ddf.columns:\n
        df_ddf = df_ddf.rename(columns={old_name: new_name})\n
        print(f\
)\n
\n
# Ensure we have 'mjd' column\n
if 'mjd' not in df_ddf.columns:\n
    if 'expMidptMJD' in df_ddf.columns:\n

In [ ]:
# Calculate skynoise from limiting magnitude\n
# skysurvey expects 'skynoise' column (inverse variance in e-/pixel)\n
\n
def calculate_skynoise_from_maglim(maglim, zp=30.0, gain=1.0):\n
    \
\
    Calculate skynoise (inverse variance) from 5-sigma limiting magnitude.\n
    \n
    Uses the formula from skysurvey.tools.utils.get_skynoise_from_maglimit:\n
    skynoise = 1.0 / (gain * 10.0 ** ((zp - m5) / 2.5))\n
    \n
    Parameters\n
    ----------\n
    maglim : array-like\n
        5-sigma limiting magnitude\n
    zp : float\n
        Zeropoint (default 30.0, skysurvey convention)\n
    gain : float\n
        Gain (default 1.0, skysurvey convention)\n
    \n
    Returns\n
    -------\n
    skynoise : array-like\n
        Inverse variance in e-/pixel\n
    \
\
    flux_5sigma = 10.0 ** ((zp - maglim) / 2.5)\n
    skynoise = 1.0 / (gain * flux_5sigma)\n
    return skynoise\n
\n
# Check what limiting magnitude columns we have\n

In [ ]:
# Standardize seeing column name\n

---\n
## 8. Prepare Final Table for skysurvey

In [ ]:
# Map columns to skysurvey naming convention\n
skysurvey_columns = {\n
    'id': 'visitId',\n
    'obs_id': 'obsId',\n
    'day_obs': 'dayObs',\n
    'seq_num': 'seqNum',\n
    'ra': 'ra',\n
    'dec': 'dec',\n
    'mjd': 'mjd',\n

    'band': 'band',\n
    'filter': 'filter',\n
    'exposure_time': 'expTime',\n

    'airmass': 'airmass',\n
    'zenith_angle': 'zenithDistance',\n
    'sky_angle': 'skyRotation',\n
    'azimuth': 'azimuth',\n

    'mean_seeing': 'mean_seeing',\n
    'mean_skyBg': 'mean_skyBg',\n
    'mean_maglim': 'mean_maglim',\n
    'fiveSigmaDepth': 'fiveSigmaDepth',\n

    'skynoise': 'skynoise',\n

    'observation_type': 'observation_type',\n
    'science_program': 'science_program',\n
    'target_name': 'target',\n

}\n
\n
# Create final table\n
df_final = df_ddf.copy()\n
\n
# Rename columns\n
for old, new in skysurvey_columns.items():\n
    if old in df_final.columns:\n
        df_final = df_final.rename(columns={old: new})\n
\n
# Add gain and zp columns (skysurvey conventions)\n
if 'gain' not in df_final.columns:\n

In [ ]:
# Show first few rows of final table\n
print(\
)\n
key_cols = ['visitId', 'band', 'ra', 'dec', 'mjd', 'expTime', 'airmass', \\\n

---\n
## 9. Save Output Files

In [ ]:
# Save as ECSV\n
print(f\
)\n
try:\n
    df_final.to_csv(OUTPUT_VISITS_ECSV, index=False)\n
    print(\
)\n
except Exception as e:\n
    print(f\
)\n
\n
# Save as Parquet (recommended for large datasets)\n
print(f\
)\n
try:\n
    df_final.to_parquet(OUTPUT_VISITS_PARQUET, index=False)\n
    print(\
)\n
except Exception as e:\n
    print(f\
)\n
\n
print(f\
)\n
print(f\
)\n
print(f\
)

In [ ]:
# Print summary statistics\n
print(\
)\n
print(f\
)\n
print(f\

---\n
## 10. Validate skysurvey Compatibility

In [ ]:
# Check required columns for skysurvey.LSST\n

---\n
## 11. Next Steps

### How to Use This Table with skysurvey\n
\n
Once you have the consolidated visit table, use it with `skysurvey`:\n
\n
```python\n
import skysurvey\n
import pandas as pd\n
\n
# Load the visit table\n
df_visits = pd.read_parquet('data_DP2_DDF_VISITS_CONSOLIDATED_01/dp2_ddf_visits_consolidated.parquet')\n
\n
# Create survey object\n
survey = skysurvey.LSST(data=df_visits)\n
\n
# Draw SN Ia population\n
from skysurvey import SNeIa\n
snia = SNeIa.from_draw(\n

In [ ]:
# Completion message\n
from datetime import datetime\n
print(\
)\n
print(\
 * 70)\n
print(\
)\n
print(f\
)\n
print(\
 * 70)\n
print(\
)\n
print(f\
)\n
print(f\
)\n
print(\
)\n
print(\
)